# Notebook 31 — Dimensionless Universality Scaling Law

This notebook takes the empirical boundary laws from Notebook 30 and rewrites them in a dimensionless form.

Main goals:
1. extract universality-boundary points as before,
2. normalize boundary coordinates relative to the baseline protocol,
3. test whether different control-direction boundaries can be compared on a shared dimensionless scale,
4. fit simple dimensionless scaling laws.

This turns the boundary description from:
- `T_c(alpha)`
- `T_c(omega_max)`

into:
- `T_c / T0`
- `omega_max / omega0`
- `alpha / alpha0`

The point is not to assume exact universality, but to test whether the control-space boundary admits a cleaner dimensionless representation.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'matplotlib', 'scipy']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar
from qutip import basis, qeye, tensor, sigmax, mesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Baseline protocol and control grids

In [ ]:
baseline = {
    'T': 26.0,
    'alpha': 0.10,
    'omega_max': 1.09,
    'delta0': 1.00,
    'p': 1.00,
    'q': 0.72,
}
V = 4.0

T_vals = np.array([20.0, 24.0, 26.0, 30.0, 34.0])
alpha_vals = np.array([0.06, 0.08, 0.10, 0.12, 0.14])
omega_vals = np.array([0.95, 1.02, 1.09, 1.16, 1.23])

T0 = baseline['T']
alpha0 = baseline['alpha']
omega0 = baseline['omega_max']

print("Baseline:", baseline)
print("Dimensionless references:", {"T0": T0, "alpha0": alpha0, "omega0": omega0})


## Shaped schedules and Hamiltonian

In [ ]:
def omega_shaped(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_shaped(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))

H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_shaped(t, T=T, omega_max=omega_max, p=p)
    def coeff_delta(t, args=None):
        return delta_shaped(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)
    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Collapse operators

In [ ]:
def collapse_ops(gamma_decay=0.0, gamma_phi=0.0):
    ops = []
    if gamma_decay > 0:
        ops.append(np.sqrt(gamma_decay) * sm1)
        ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        ops.append(np.sqrt(gamma_phi) * n1)
        ops.append(np.sqrt(gamma_phi) * n2)
    return ops


## Noisy evolution and effective map

In [ ]:
def run_noisy_evolution(psi0, params, gamma_decay=0.0, gamma_phi=0.0, n_steps=140):
    H = build_time_dependent_hamiltonian(
        params['T'], params['omega_max'], params['alpha'], V, params['delta0'], params['p'], params['q']
    )
    times = np.linspace(0.0, params['T'], n_steps)
    c_ops = collapse_ops(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    result = mesolve(H, psi0, times, c_ops)
    return result.states[-1]

def state_to_column_mixed(state):
    vals = []
    for basis_state in basis_states:
        if state.isket:
            vals.append(basis_state.overlap(state))
        else:
            val = basis_state.dag() * state * basis_state
            vals.append(np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val))
    return np.array(vals, dtype=complex)

def build_noisy_effective_map(params, gamma_decay=0.0, gamma_phi=0.0, n_steps=140):
    cols = []
    finals = []
    for psi0 in basis_states:
        final_state = run_noisy_evolution(
            psi0, params, gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=n_steps
        )
        cols.append(state_to_column_mixed(final_state))
        finals.append(final_state)
    return np.column_stack(cols), finals


## Diagnostics

In [ ]:
def process_fidelity(U_eff, U_target):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def extract_local_z_phases(U_eff):
    phi00, phi01, phi10, phi11 = diagonal_phases(U_eff)
    phi_ent = entangling_phase(U_eff)
    a = wrapped_phase(phi10 - phi00)
    b = wrapped_phase(phi01 - phi00)
    global_phase = phi00
    return global_phase, a, b, phi_ent

def local_z_diagonal(a, b):
    return np.diag([1.0, np.exp(1j*b), np.exp(1j*a), np.exp(1j*(a+b))]).astype(complex)

def compensate_local_z(U_eff):
    global_phase, a, b, phi_ent = extract_local_z_phases(U_eff)
    U1 = np.exp(-1j * global_phase) * U_eff
    U2 = U1 @ local_z_diagonal(-a, -b)
    return U2, global_phase, a, b, phi_ent

def compensated_cz_fidelity(U_eff):
    U_comp, _, _, _, _ = compensate_local_z(U_eff)
    return process_fidelity(U_comp, U_cz)

def leakage_from_finals(final_states):
    scores = []
    for idx, final_state in enumerate(final_states):
        basis_state = basis_states[idx]
        if final_state.isket:
            score = np.abs(basis_state.overlap(final_state)) ** 2
        else:
            val = basis_state.dag() * final_state * basis_state
            score = np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val)
        scores.append(score)
    return float(1.0 - np.mean(scores))

def coherence_proxy(final_states):
    vals = []
    for state in final_states:
        rho = state.proj() if state.isket else state
        arr = rho.full()
        off = arr.copy()
        np.fill_diagonal(off, 0.0)
        vals.append(np.mean(np.abs(off)))
    return float(np.mean(vals))


## Shared noise grid

In [ ]:
gamma_decay_vals = np.linspace(0.0, 0.12, 13)
gamma_phi_vals = np.linspace(0.0, 0.12, 13)


## Helper: evaluate one protocol

In [ ]:
def evaluate_protocol(params):
    cz_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
    coh_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
    leak_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))

    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            U_eff, finals = build_noisy_effective_map(
                params, gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=140
            )
            cz_map[i, j] = compensated_cz_fidelity(U_eff)
            coh_map[i, j] = coherence_proxy(finals)
            leak_map[i, j] = leakage_from_finals(finals)

    S = cz_map * coh_map * (1.0 - leak_map)
    S_norm = S / np.max(S) if np.max(S) > 0 else S.copy()

    S_gamma = S_norm[0, :]
    S_phi = S_norm[:, 0]
    interp_gamma = interp1d(gamma_decay_vals, S_gamma, kind='linear', fill_value='extrapolate')

    def loss(lam):
        mapped = lam * gamma_phi_vals
        pred = interp_gamma(np.clip(mapped, gamma_decay_vals.min(), gamma_decay_vals.max()))
        return float(np.mean((pred - S_phi) ** 2))

    fit = minimize_scalar(loss, bounds=(0.1, 5.0), method='bounded')
    lam = float(fit.x)
    axis_loss = float(fit.fun)

    gamma_eff_grid = np.zeros_like(S_norm)
    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            gamma_eff_grid[i, j] = gamma_decay + lam * gamma_phi

    gamma_eff_flat = gamma_eff_grid.ravel()
    S_flat = S_norm.ravel()
    order = np.argsort(gamma_eff_flat)
    ge = gamma_eff_flat[order]
    sv = S_flat[order]

    n_bins = 12
    bins = np.linspace(ge.min(), ge.max(), n_bins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    means = np.full(n_bins, np.nan)

    for k in range(n_bins):
        mask = (ge >= bins[k]) & (ge < bins[k+1])
        if np.any(mask):
            means[k] = np.mean(sv[mask])

    valid = ~np.isnan(means)
    x = centers[valid]
    y = means[valid]

    return {
        'params': params,
        'lambda': lam,
        'axis_loss': axis_loss,
        'curve_x': x,
        'curve_y': y,
    }


## Baseline reference

In [ ]:
baseline_res = evaluate_protocol(baseline)
lambda0 = baseline_res['lambda']
curve0_x = baseline_res['curve_x']
curve0_y = baseline_res['curve_y']
interp0 = interp1d(curve0_x, curve0_y, kind='linear', fill_value='extrapolate')

print("baseline lambda =", lambda0)


## Universality score

In [ ]:
def curve_mismatch_to_baseline(x, y):
    y_ref = interp0(np.clip(x, curve0_x.min(), curve0_x.max()))
    return float(np.mean((y - y_ref) ** 2))

def universality_score(lambda_shift, axis_loss, curve_mismatch,
                       lambda_scale=0.15, loss_scale=0.001, curve_scale=0.001):
    return float(np.exp(
        -(lambda_shift / lambda_scale)
        -(axis_loss / loss_scale)
        -(curve_mismatch / curve_scale)
    ))

boundary_level = 0.30
print("boundary level =", boundary_level)


## Evaluate score maps

In [ ]:
TA_score = np.zeros((len(alpha_vals), len(T_vals)))
TO_score = np.zeros((len(omega_vals), len(T_vals)))

for i, alpha in enumerate(alpha_vals):
    for j, T in enumerate(T_vals):
        params = {**baseline, 'T': float(T), 'alpha': float(alpha)}
        res = evaluate_protocol(params)
        lambda_shift = abs(res['lambda'] - lambda0)
        curve_mismatch = curve_mismatch_to_baseline(res['curve_x'], res['curve_y'])
        TA_score[i, j] = universality_score(lambda_shift, res['axis_loss'], curve_mismatch)

for i, omega in enumerate(omega_vals):
    for j, T in enumerate(T_vals):
        params = {**baseline, 'T': float(T), 'omega_max': float(omega)}
        res = evaluate_protocol(params)
        lambda_shift = abs(res['lambda'] - lambda0)
        curve_mismatch = curve_mismatch_to_baseline(res['curve_x'], res['curve_y'])
        TO_score[i, j] = universality_score(lambda_shift, res['axis_loss'], curve_mismatch)


## Boundary extraction

In [ ]:
def extract_vertical_boundary(score_map, xvals, yvals, level):
    bx, by = [], []
    for i, y in enumerate(yvals):
        row = score_map[i, :]
        crossing = None
        for j in range(len(xvals) - 1):
            v0, v1 = row[j], row[j + 1]
            if (v0 >= level and v1 < level) or (v0 > level and v1 <= level):
                x0, x1 = xvals[j], xvals[j + 1]
                xc = x0 if v1 == v0 else x0 + (level - v0) * (x1 - x0) / (v1 - v0)
                crossing = xc
                break
        if crossing is None and np.max(row) >= level and np.min(row) >= level:
            crossing = float(xvals[-1])
        if crossing is not None:
            bx.append(float(crossing))
            by.append(float(y))
    return np.array(bx), np.array(by)

TA_bx, TA_by = extract_vertical_boundary(TA_score, T_vals, alpha_vals, boundary_level)
TO_bx, TO_by = extract_vertical_boundary(TO_score, T_vals, omega_vals, boundary_level)

print("T-alpha boundary points:", list(zip(TA_bx, TA_by)))
print("T-omega boundary points:", list(zip(TO_bx, TO_by)))


## Dimensionless boundary coordinates

In [ ]:
TA_x = TA_by / alpha0     # alpha / alpha0
TA_y = TA_bx / T0         # T_c / T0

TO_x = TO_by / omega0     # omega / omega0
TO_y = TO_bx / T0         # T_c / T0

print("Dimensionless T-alpha points:", list(zip(TA_x, TA_y)))
print("Dimensionless T-omega points:", list(zip(TO_x, TO_y)))


## Simple dimensionless model fits

In [ ]:
def fit_constant(y, x):
    c = float(np.mean(y))
    yhat = np.full_like(y, c, dtype=float)
    return {'name': 'constant', 'coeffs': [c], 'yhat': yhat}

def fit_linear(y, x):
    coeffs = np.polyfit(x, y, deg=1)
    yhat = np.polyval(coeffs, x)
    return {'name': 'linear', 'coeffs': coeffs.tolist(), 'yhat': yhat}

def fit_quadratic(y, x):
    coeffs = np.polyfit(x, y, deg=2)
    yhat = np.polyval(coeffs, x)
    return {'name': 'quadratic', 'coeffs': coeffs.tolist(), 'yhat': yhat}

def fit_power_law_shifted(y, x):
    # Fit y ≈ a + b*(x-1)
    coeffs = np.polyfit(x - 1.0, y, deg=1)
    yhat = np.polyval(coeffs, x - 1.0)
    return {'name': 'shifted_linear', 'coeffs': coeffs.tolist(), 'yhat': yhat}

def score_fit(y, yhat):
    mse = float(np.mean((y - yhat) ** 2))
    mae = float(np.mean(np.abs(y - yhat)))
    var = float(np.var(y))
    r2 = float(1.0 - mse / var) if var > 0 else np.nan
    return {'mse': mse, 'mae': mae, 'r2': r2}

def fit_models(x, y):
    fits = []
    for fitter in [fit_constant, fit_linear, fit_quadratic, fit_power_law_shifted]:
        fit = fitter(y, x)
        fit.update(score_fit(y, fit['yhat']))
        fits.append(fit)
    return fits

TA_fits = fit_models(TA_x, TA_y) if len(TA_x) >= 3 else []
TO_fits = fit_models(TO_x, TO_y) if len(TO_x) >= 3 else []


## Dimensionless scaling law plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

if TA_fits:
    axes[0].scatter(TA_x, TA_y, label='extracted boundary')
    xfine = np.linspace(TA_x.min(), TA_x.max(), 200)
    for fit in TA_fits:
        coeffs = fit['coeffs']
        if fit['name'] == 'constant':
            yfine = np.full_like(xfine, coeffs[0], dtype=float)
        elif fit['name'] == 'shifted_linear':
            yfine = np.polyval(coeffs, xfine - 1.0)
        else:
            yfine = np.polyval(coeffs, xfine)
        axes[0].plot(xfine, yfine, label=fit['name'])
    axes[0].axvline(1.0, linestyle='--', alpha=0.6)
    axes[0].axhline(1.0, linestyle='--', alpha=0.6)
    axes[0].set_xlabel('alpha / alpha0')
    axes[0].set_ylabel('T_c / T0')
    axes[0].set_title('Dimensionless scaling candidates for T_c(alpha)')
    axes[0].legend(fontsize=8)

if TO_fits:
    axes[1].scatter(TO_x, TO_y, label='extracted boundary')
    xfine = np.linspace(TO_x.min(), TO_x.max(), 200)
    for fit in TO_fits:
        coeffs = fit['coeffs']
        if fit['name'] == 'constant':
            yfine = np.full_like(xfine, coeffs[0], dtype=float)
        elif fit['name'] == 'shifted_linear':
            yfine = np.polyval(coeffs, xfine - 1.0)
        else:
            yfine = np.polyval(coeffs, xfine)
        axes[1].plot(xfine, yfine, label=fit['name'])
    axes[1].axvline(1.0, linestyle='--', alpha=0.6)
    axes[1].axhline(1.0, linestyle='--', alpha=0.6)
    axes[1].set_xlabel('omega_max / omega0')
    axes[1].set_ylabel('T_c / T0')
    axes[1].set_title('Dimensionless scaling candidates for T_c(omega_max)')
    axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


## Overlay dimensionless boundaries

In [ ]:
plt.figure(figsize=(7.2, 5.0))
if len(TA_x) > 0:
    plt.scatter(TA_x, TA_y, label='T_c(alpha) / baseline')
if len(TO_x) > 0:
    plt.scatter(TO_x, TO_y, label='T_c(omega_max) / baseline')
plt.axvline(1.0, linestyle='--', alpha=0.6)
plt.axhline(1.0, linestyle='--', alpha=0.6)
plt.xlabel('dimensionless control variable')
plt.ylabel('T_c / T0')
plt.title('Dimensionless universality-boundary points')
plt.legend()
plt.tight_layout()
plt.show()


## Residual plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

plots = [
    ('T_c(alpha)', TA_x, TA_y, TA_fits),
    ('T_c(omega_max)', TO_x, TO_y, TO_fits),
]

for row, (label, x, y, fits) in enumerate(plots):
    for col in range(2):
        ax = axes[2*row + col]
        if not fits:
            ax.set_visible(False)
            continue
        fit = fits[col] if col < len(fits) else fits[0]
        resid = y - fit['yhat']
        ax.axhline(0.0, linestyle='--', alpha=0.7)
        ax.scatter(x, resid)
        ax.set_title(f"{label}: {fit['name']} residuals")
        ax.set_xlabel('dimensionless control variable')
        ax.set_ylabel('T_c / T0 residual')

plt.tight_layout()
plt.show()


## Fit summaries

In [ ]:
def print_fit_summary(label, fits):
    print(label)
    if not fits:
        print("  not enough boundary points")
        return
    for fit in fits:
        coeffs_str = ", ".join([f"{c:.6g}" for c in fit['coeffs']])
        print(
            f"  {fit['name']}: coeffs=[{coeffs_str}], "
            f"MSE={fit['mse']:.6e}, MAE={fit['mae']:.6e}, R2={fit['r2']:.6f}"
        )

print_fit_summary("Dimensionless T_c(alpha)", TA_fits)
print()
print_fit_summary("Dimensionless T_c(omega_max)", TO_fits)


## Compact summary

In [ ]:
lines = []
lines.append("Dimensionless universality scaling law")
lines.append("")
lines.append(f"Baseline values: T0={T0:.4f}, alpha0={alpha0:.4f}, omega0={omega0:.4f}")
lines.append(f"Boundary score level = {boundary_level:.2f}")
lines.append("")

if TA_fits:
    best_TA = sorted(TA_fits, key=lambda d: d['mse'])[0]
    lines.append(
        f"Best dimensionless T_c(alpha) model: {best_TA['name']} "
        f"(MSE={best_TA['mse']:.6e}, R2={best_TA['r2']:.6f})"
    )
else:
    lines.append("No dimensionless T_c(alpha) fit available.")

if TO_fits:
    best_TO = sorted(TO_fits, key=lambda d: d['mse'])[0]
    lines.append(
        f"Best dimensionless T_c(omega_max) model: {best_TO['name']} "
        f"(MSE={best_TO['mse']:.6e}, R2={best_TO['r2']:.6f})"
    )
else:
    lines.append("No dimensionless T_c(omega_max) fit available.")

lines.append("")
lines.append("Interpretation:")
lines.append("- If the dimensionless alpha boundary is nearly flat, alpha enters weakly in the boundary law.")
lines.append("- If the dimensionless omega boundary is well fit by a shifted linear law, omega_max sets the leading deformation of the universality edge.")
lines.append("- The dimensionless form makes it easier to compare boundary structure across protocol families.")

summary_text = "\n".join(lines)
print(summary_text)


## Final conclusion

This notebook rewrites the universality boundary in dimensionless form and tests whether simple low-parameter scaling laws survive normalization to the baseline protocol.

That gives the cleanest next-level summary:

**the projection-stable phase-locked CZ regime admits not only an explicit boundary in control space, but also a dimensionless scaling representation that helps distinguish weak control directions from genuinely boundary-shaping ones.**
